## Syn Bank Wallet Intelligence — Pipeline

Reproducible pipeline from raw data to the consolidated client table. Cleaning and 
aggregation rules applied here were derived and justified in `01_eda.ipynb` — see that 
notebook for the reasoning behind each decision.

**Scope:** analysis covers the 20 clients (E01–E20) present in the provided datasets. 
The brief describes a 50-client portfolio; no data was provided for the remaining 30 
(E21–E50), so this pipeline and all downstream outputs are scoped to the 20 observed 
clients.

In [1]:
import pandas as pd

cross_border_payments = pd.read_csv("../data/cross_border_payments.csv")
trade_finance = pd.read_csv("../data/trade_finance.csv")
transactional_banking = pd.read_csv("../data/transactional_banking.csv")

# Currency casing fix (found during EDA: 'ZAR' vs 'zar')
transactional_banking['currency'] = transactional_banking['currency'].str.upper()

print(cross_border_payments.shape, transactional_banking.shape, trade_finance.shape)

(241117, 13) (2802875, 13) (20303, 15)


## Consolidated per-client table

One row per client (E01–E20), with pillar totals from each dataset. Design choice 
(from EDA): lending/Investment-Banking-flavoured transactions — identified via a 
populated `memo` field — are pulled out of their pillar and summed separately, so each 
pillar total reflects "pure" activity for that product type rather than blending in 
financing-related transactions.

Exclusions applied per pillar (justified in EDA):
- Transactional banking: excludes `intercompany_sweeps` (internal treasury movement)
- Cross-border payments: excludes `intercompany` corridor (internal treasury movement)
- Trade finance: no exclusions beyond the memo split

In [2]:
# --- Base client info ---
clients = (
    transactional_banking[['entity_id', 'entity_name', 'sector']]
    .drop_duplicates()
    .sort_values('entity_id')
    .reset_index(drop=True)
)

# --- Transactional banking: exclude intercompany_sweeps and memo-tagged rows ---
tb_lending_mask = transactional_banking['memo'].notna()
tb_core = transactional_banking[
    (transactional_banking['leg_type'] != 'intercompany_sweeps')
    & (~tb_lending_mask)
]
tb_totals = (
    tb_core.groupby('entity_id')['amount_zar']
    .sum()
    .rename('txn_banking_total_zar')
)

# --- Cross-border: exclude intercompany corridor, exclude memo-tagged rows ---
cb_lending_mask = cross_border_payments['memo'].notna()
cb_core = cross_border_payments[
    (cross_border_payments['corridor_type'] != 'intercompany')
    & (~cb_lending_mask)
]
cb_totals = (
    cb_core.groupby('entity_id')['value_zar']
    .sum()
    .rename('cross_border_total_zar')
)

# --- Trade finance: exclude memo-tagged rows ---
tf_lending_mask = trade_finance['memo'].notna()
tf_core = trade_finance[~tf_lending_mask]
tf_totals = (
    tf_core.groupby('entity_id')['value_zar']
    .sum()
    .rename('trade_finance_total_zar')
)

# --- Lending / Investment Banking signal, combined across all 3 datasets ---
lending_parts = [
    transactional_banking.loc[tb_lending_mask, ['entity_id', 'amount_zar']]
        .rename(columns={'amount_zar': 'value_zar'}),
    cross_border_payments.loc[cb_lending_mask, ['entity_id', 'value_zar']],
    trade_finance.loc[tf_lending_mask, ['entity_id', 'value_zar']],
]
lending_combined = pd.concat(lending_parts, ignore_index=True)

lending_totals = (
    lending_combined.groupby('entity_id')['value_zar']
    .agg(lending_signal_total_zar='sum', lending_signal_txn_count='count')
)

# --- Merge onto client base table ---
client_table = (
    clients
    .merge(tb_totals, on='entity_id', how='left')
    .merge(cb_totals, on='entity_id', how='left')
    .merge(tf_totals, on='entity_id', how='left')
    .merge(lending_totals, on='entity_id', how='left')
)

value_cols = [
    'txn_banking_total_zar', 'cross_border_total_zar',
    'trade_finance_total_zar', 'lending_signal_total_zar', 'lending_signal_txn_count'
]
client_table[value_cols] = client_table[value_cols].fillna(0)

client_table['syn_bank_observed_total_zar'] = (
    client_table['txn_banking_total_zar']
    + client_table['cross_border_total_zar']
    + client_table['trade_finance_total_zar']
    + client_table['lending_signal_total_zar']
)

client_table.sort_values('syn_bank_observed_total_zar', ascending=False)

,entity_id,entity_name,sector,txn_banking_total_zar,cross_border_total_zar,trade_finance_total_zar,lending_signal_total_zar,lending_signal_txn_count,syn_bank_observed_total_zar
10,E11,Pepkor Holdings,consumer,4.789290e+10,1.095877e+10,4.539648e+09,0.000000e+00,0.0,6.339132e+10
7,E08,Sanlam,insurance,3.427667e+10,4.607626e+09,5.800268e+08,0.000000e+00,0.0,3.946433e+10
0,E01,BHP Group,mining,3.175649e+10,3.484143e+09,3.949315e+09,0.000000e+00,0.0,3.918995e+10
15,E16,MTN Group,telecoms,1.347148e+10,1.060554e+10,4.391391e+09,2.269990e+08,844.0,2.869541e+10
9,E10,Bid Corporation,consumer,1.443446e+10,8.330591e+09,5.461465e+09,2.068692e+08,881.0,2.843338e+10
8,E09,Shoprite Holdings,consumer,1.336835e+10,5.433298e+09,4.260904e+09,1.753583e+08,793.0,2.323791e+10
2,E03,Anglo American,mining,1.795186e+10,2.279056e+09,2.339380e+09,0.000000e+00,0.0,2.257030e+10
1,E02,Glencore,mining,8.304405e+09,3.808527e+09,3.104991e+09,5.500265e+07,83.0,1.527293e+10
17,E18,The Bidvest Group,industrials_pharma,5.777326e+09,2.753695e+09,3.753096e+09,1.012459e+08,409.0,1.238536e+10
18,E19,Aspen Pharmacare,industrials_pharma,4.105327e+09,2.367702e+09,1.292890e+09,5.987651e+07,408.0,7.825797e+09
